# Bộ next-step theo nhóm timestamp
Một mẫu = prefix từ đầu phiên đến t; target = tập item tại nhóm kế tiếp cùng phiên, mọi hành vi. Đầu ra đã tạo trên toàn RetailRocket.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/nextstep.json').is_file())
sys.path.insert(0, str(ROOT))
from IPython.display import display
from scripts.dataset import require_complete, history_for_query, candidates_at_time
folder = require_complete(ROOT / 'data/nextstep')
config = json.loads((ROOT / 'configs/nextstep.json').read_text())
summary = json.loads((ROOT / 'reports/nextstep/summary.json').read_text())
display(pd.DataFrame(summary['split_summary']).T[['queries','sessions','nextstep_target_pairs']])

,queries,sessions,nextstep_target_pairs
train,806025,306332,807908
validation,92566,38290,92794
test,92468,37864,92634


In [2]:
queries = pd.read_parquet(folder / 'queries.parquet')
targets = pd.read_parquet(folder / 'nextstep_targets.parquet')
events = pd.read_parquet(folder / 'events.parquet')
query = queries.loc[queries.split.eq('test')].iloc[0]
print('Real query:', query.query_id, 'session:', query.session_id)
prefix = history_for_query(events, query)
truth = targets[targets.query_id.eq(query.query_id)]
display(prefix[['session_id','event_time','itemid','event']])
display(truth[['query_id','itemid','target_time','target_event_count','candidate_at_query']])

Real query: 1 session: 1


,session_id,event_time,itemid,event
0,1,2015-09-11 20:49:49.439000+00:00,285930,view


,query_id,itemid,target_time,target_event_count,candidate_at_query
0,1,357564,2015-09-11 20:52:39.591000+00:00,1,True


## Tách thông tin quan sát và nhãn
Không đưa `target_time`, `target_count`, `eligible_target_count` hay danh sách item đúng vào feature hoặc tập ứng viên. Cho phép target lặp item trong prefix.

In [3]:
assert prefix.session_id.eq(query.session_id).all()
assert prefix.event_time.le(query.query_time).all()
assert truth.target_time.eq(query.target_time).all()
assert query.target_time > query.query_time
print('Session/time boundary checks passed')

Session/time boundary checks passed


In [4]:
candidates = candidates_at_time(folder, query.query_time, config)
print('Eligible catalog items at query:', len(candidates))
display(candidates[['itemid','known_at','available','category_id']].head())

Eligible catalog items at query: 101844


,itemid,known_at,available,category_id
0,9,2015-05-13 22:53:14.505000+00:00,<NA>,-1
1,19,2015-05-03 21:46:12.361000+00:00,1,1171
2,22,2015-05-10 03:00:00+00:00,1,1359
3,25,2015-05-08 20:54:13.145000+00:00,1,72
4,37,2015-06-27 23:43:42.544000+00:00,<NA>,-1


## Phạm vi hiện tại
Pipeline, kiểm định và dữ liệu đã hoàn tất. Chưa có mô hình, checkpoint, chỉ số chất lượng xếp hạng hoặc demo. Độ phủ ứng viên trong summary không phải Recall/NDCG của mô hình.